# Publication Figure: Stoichiometric Profile derived from excess kinetics
Figure showing the dependency of stoichiometric yield coefficients as function of the glucose and glutamine concentration.

In [7]:
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add package source path
repo_root = Path.cwd().parent
src_path = repo_root / "animakinetica" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from animakinetica.core.parameters import SimulationParameters
from animakinetica.models.kinetics import Kinetics, create_kinetic_model
from animakinetica.models.stoichiometry import StoichiometricModel, StoichiometryMode

# --- Fixed conditions ---
C_Gln_fixed = 1.0  # mmol/L
C_Glc_range = np.linspace(0.0, 50.0, 300)

# --- Parameter sets ---
# GFI stoichiometry
params_gfi = SimulationParameters.load_parameters([
    "base_parameters",
    "stoichiometry/anab_GFI",
    "stoichiometry/cata_GFI",
    "cell_properties/cell_GFI",
])

# Humbird normal stoichiometry
params_humbird_normal = SimulationParameters.load_parameters([
    "base_parameters",
    "stoichiometry/anab_Humbird_cell",
    "stoichiometry/cata_Humbird_wildtype",
    "cell_properties/cell_Humbird",
])

# Humbird enhanced stoichiometry
params_humbird_enhanced = SimulationParameters.load_parameters([
    "base_parameters",
    "stoichiometry/anab_Humbird_cell",
    "stoichiometry/cata_Humbird_enhanced",
    "cell_properties/cell_Humbird",
])

# Excess kinetics (normal)
params_exc_normal = SimulationParameters.load_parameters([
    "base_parameters",
    "excess_kinetics/excess_kinetics_batch",
    "stoichiometry/anab_GFI",
    "cell_properties/cell_GFI",
])

# Excess kinetics (adjusted)
params_exc_adjusted = SimulationParameters.load_parameters([
    "base_parameters",
    "excess_kinetics/excess_kinetics_adjusted",
    "stoichiometry/anab_GFI",
    "cell_properties/cell_GFI",
])

# Use GFI mu_max as reference for plotting condition
mu_max = float(params_gfi.cell_properties.mu_max)
mu_fixed = 0.8 * mu_max

# --- Build stoichiometric models ---
# Constant + Dynamic for each stoichiometric family
stoich_sets = {
    "GFI": params_gfi,
    "Humbird normal": params_humbird_normal,
    "Humbird enhanced": params_humbird_enhanced,
}

constant_yields = {}
dynamic_yields = {}
for set_name, params_local in stoich_sets.items():
    model_const = StoichiometricModel(params_local, mode=StoichiometryMode.CONSTANT)
    model_dyn = StoichiometricModel(params_local, mode=StoichiometryMode.DYNAMIC)
    constant_yields[set_name] = model_const.yields.copy()
    dynamic_yields[set_name] = model_dyn.calculate_yields(mu_fixed).copy()

# Excess-coupled model with normal excess kinetics
model_excess_normal = StoichiometricModel(params_exc_normal, mode=StoichiometryMode.EXCESS_COUPLED)
kin_excess_normal = create_kinetic_model(Kinetics.EXCESS, params_exc_normal, stoichiometric_model=model_excess_normal)

excess_normal_yields_matrix = np.array([
    model_excess_normal.calculate_yields(
        mu_fixed,
        cat_yields=kin_excess_normal.calculate_cat_yields_from_excess(mu_fixed, c_glc, C_Gln_fixed),
    )
    for c_glc in C_Glc_range
])

# Excess-coupled model with adjusted excess kinetics
model_excess_adj = StoichiometricModel(params_exc_adjusted, mode=StoichiometryMode.EXCESS_COUPLED)
kin_excess_adj = create_kinetic_model(Kinetics.EXCESS, params_exc_adjusted, stoichiometric_model=model_excess_adj)

excess_adjusted_yields_matrix = np.array([
    model_excess_adj.calculate_yields(
        mu_fixed,
        cat_yields=kin_excess_adj.calculate_cat_yields_from_excess(mu_fixed, c_glc, C_Gln_fixed),
    )
    for c_glc in C_Glc_range
])

# Yield indices to highlight
yield_panels = [
    ("Y<sub>Glc/X</sub>", 0),
    ("Y<sub>Lac/Glc</sub>", 1),
    ("Y<sub>Gln/Glc</sub>", 2),
    ("Y<sub>Amm/Gln</sub>", 6),
]

# --- Publication figure ---
fig = make_subplots(
    rows=2,
    cols=2,
    horizontal_spacing=0.10,
    vertical_spacing=0.12,
)

# CVD-friendly colors
set_colors = {
    "Humbird normal": "#D55E00",
    "Humbird enhanced": "#8C3B00",
    "GFI": "#0072B2",
    "GFI_dark": "#2ca02c",
}

name_1 = "1. Humbird (2020) - Wild-Type"
name_2 = "2. Humbird (2020) - Enhanced"
name_3 = "3. Frohlich et al. (2025)"
# name_4 = "4. Growth rate dependent"
name_5 = "4. Substrate Excess - Zeng & Deckwer (1995)"
name_6 = "5. Substrate Excess - Enhanced"

constant_trace_names = {
    "Humbird normal": name_1,
    "Humbird enhanced": name_2,
    "GFI": name_3,
}

# Row-wise legend draw order
legend_rank = {
    name_1: 1,
    name_3: 2,
    name_5: 5,
    name_2: 4,
    # name_4: 5,
    name_6: 3,
}

linewidth = 8
axis_line_width = 3.5
tick_width = 2.5

for i, (label, idx) in enumerate(yield_panels):
    row = i // 2 + 1
    col = i % 2 + 1

    # Constant traces for all parameter families (solid)
    for set_name in ["Humbird normal", "Humbird enhanced", "GFI"]:
        color = set_colors[set_name]
        trace_name = constant_trace_names[set_name]

        fig.add_trace(
            go.Scatter(
                x=C_Glc_range,
                y=np.full_like(C_Glc_range, constant_yields[set_name][idx]),
                mode="lines",
                name=trace_name,
                legendrank=legend_rank[trace_name],
                line=dict(color=color, width=linewidth, dash="solid"),
                showlegend=(i == 0),
            ),
            row=row,
            col=col,
        )

    # Dynamic trace only for GFI (dashed) - Campaign 3 excluded
    # fig.add_trace(
    #     go.Scatter(
    #         x=C_Glc_range,
    #         y=np.full_like(C_Glc_range, dynamic_yields["GFI"][idx]),
    #         mode="lines",
    #         name=name_4,
    #         legendrank=legend_rank[name_4],
    #         line=dict(color=set_colors["GFI"], width=linewidth, dash="dash"),
    #         showlegend=(i == 0),
    #     ),
    #     row=row,
    #     col=col,
    # )

    # Excess-coupled with normal excess kinetics (dotted)
    fig.add_trace(
        go.Scatter(
            x=C_Glc_range,
            y=excess_normal_yields_matrix[:, idx],
            mode="lines",
            name=name_5,
            legendrank=legend_rank[name_5],
            line=dict(color=set_colors["GFI"], width=linewidth, dash="dot"),
            showlegend=(i == 0),
        ),
        row=row,
        col=col,
    )

    # Excess-coupled with adjusted excess kinetics (dotted)
    fig.add_trace(
        go.Scatter(
            x=C_Glc_range,
            y=excess_adjusted_yields_matrix[:, idx],
            mode="lines",
            name=name_6,
            legendrank=legend_rank[name_6],
            line=dict(color=set_colors["GFI_dark"], width=linewidth, dash="dot"),
            showlegend=(i == 0),
        ),
        row=row,
        col=col,
    )

    fig.update_xaxes(title_text="C<sub>Glc</sub> (mmol/L)", row=row, col=col)
    fig.update_yaxes(title_text=label, row=row, col=col)

fig.update_xaxes(
    showline=True,
    linewidth=axis_line_width,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=tick_width,
)
fig.update_yaxes(
    showline=True,
    linewidth=axis_line_width,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=tick_width,
)

fig.update_layout(
    width=1700,
    height=1700,
    template="simple_white",
    font=dict(size=35, color="black"),
    legend=dict(
        title_text="",
        orientation="h",
        yanchor="top",
        y=-0.12,
        xanchor="left",
        x=0.0,
        traceorder="normal",
        entrywidthmode="fraction",
        entrywidth=1 / 3,
        font=dict(size=20),
    ),
    margin=dict(l=100, r=0, t=0, b=280),
)

fig.show()


# Update to transparent background
fig.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",   # whole canvas
    plot_bgcolor="rgba(0,0,0,0)",    # plotting area
    legend=dict(bgcolor="rgba(0,0,0,0)")
)

# -----------------------------------------------------------------------------
# 5) Export for publication/reproducibility
# -----------------------------------------------------------------------------
SAVE_EXPORTS = True

if SAVE_EXPORTS == True:

    out_dir = Path.cwd()
    # csv_path = out_dir / "fedbatch_strategy_comparison_metrics.csv"
    svg_path = out_dir / "Kinetic_Profiles_Glc.svg"
    # png_path = out_dir / "fedbatch_strategy_comparison.png"

    # metrics_df.to_csv(csv_path, index=False)
    # print(f"Saved metrics table: {csv_path}")

    try:
        fig.write_image(str(svg_path))
        # fig.write_image(str(png_path), scale=3)
        print(f"Saved figure: {svg_path}")
        # print(f"Saved figure: {png_path}")
    except Exception as exc:
        print("Image export skipped (install/enable kaleido if needed).")
        print(f"Export error: {exc}")

Loading parameters from 4 file(s) sequentially...
  [1/4] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [3/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [4/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\cell_properties\cell_GFI.yaml
Successfully loaded parameters from 4 file(s).
Loading parameters from 4 file(s) sequentially...
  [1/4] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/4] Updating with: c:\Users\btumulero\OneDri

Saved figure: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Method Figures\Kinetic_Profiles_Glc.svg


**Figure 4.** Stoichiometric yield coefficients are shown as functions of glucose concentration for fictional animal cell-line model variants with distinct kinetic/stoichiometric behavior at fixed $C_{\mathrm{Gln}}=1.0\,\mathrm{mmol\,L^{-1}}$ and $\mu=0.8\,\mu_{\max}$; panels A–D report $Y_{\mathrm{Glc}/X}$, $Y_{\mathrm{Lac}/\mathrm{Glc}}$, $Y_{\mathrm{Gln}/\mathrm{Glc}}$, and $Y_{\mathrm{Amm}/\mathrm{Gln}}$, respectively, comparing constant stoichiometric models for Humbird (2020), Humbird (2020) Enhanced, and Frohlich et al. (2025), together with dynamic and excess-based Frohlich et al. (2025) formulations derived from Zeng & Deckwer (1995); line styles indicate model class (solid: constant, dashed: dynamic, dotted: excess).

## Same figures as function of glutamine concentration

### Glutamine sweep at fixed glucose concentration
Same 2x2 publication-style figure, now with $C_{\mathrm{Gln}}$ on the x-axis and fixed $C_{\mathrm{Glc}} = 10\,\mathrm{mmol\,L^{-1}}$.

In [8]:
# --- Fixed conditions for glutamine sweep ---
C_Glc_fixed = 20.0  # mmol/L
C_Gln_range = np.linspace(0.0, 10.0, 100)

# Use same fixed growth rate reference
mu_fixed = 0.8 * float(params_gfi.cell_properties.mu_max)

# Constant and dynamic stoichiometric yields do not vary with concentration in these modes
constant_yields_gln = {}
dynamic_yields_gln = {}
for set_name, params_local in stoich_sets.items():
    model_const = StoichiometricModel(params_local, mode=StoichiometryMode.CONSTANT)
    model_dyn = StoichiometricModel(params_local, mode=StoichiometryMode.DYNAMIC)
    constant_yields_gln[set_name] = model_const.yields.copy()
    dynamic_yields_gln[set_name] = model_dyn.calculate_yields(mu_fixed).copy()

# Excess-coupled model with normal excess kinetics (vary glutamine)
model_excess_normal_gln = StoichiometricModel(params_exc_normal, mode=StoichiometryMode.EXCESS_COUPLED)
kin_excess_normal_gln = create_kinetic_model(
    Kinetics.EXCESS,
    params_exc_normal,
    stoichiometric_model=model_excess_normal_gln,
)

excess_normal_yields_matrix_gln = np.array([
    model_excess_normal_gln.calculate_yields(
        mu_fixed,
        cat_yields=kin_excess_normal_gln.calculate_cat_yields_from_excess(mu_fixed, C_Glc_fixed, c_gln),
    )
    for c_gln in C_Gln_range
])

# Excess-coupled model with adjusted excess kinetics (vary glutamine)
model_excess_adj_gln = StoichiometricModel(params_exc_adjusted, mode=StoichiometryMode.EXCESS_COUPLED)
kin_excess_adj_gln = create_kinetic_model(
    Kinetics.EXCESS,
    params_exc_adjusted,
    stoichiometric_model=model_excess_adj_gln,
)

excess_adjusted_yields_matrix_gln = np.array([
    model_excess_adj_gln.calculate_yields(
        mu_fixed,
        cat_yields=kin_excess_adj_gln.calculate_cat_yields_from_excess(mu_fixed, C_Glc_fixed, c_gln),
    )
    for c_gln in C_Gln_range
])

# Same yield indices as previous figure
yield_panels = [
    ("Y<sub>Glc</sub>/X", 0),
    ("Y<sub>Lac</sub>/Glc", 1),
    ("Y<sub>Gln</sub>/Glc", 2),
    ("Y<sub>Amm</sub>/Gln", 6),
]

fig_gln = make_subplots(
    rows=2,
    cols=2,
    horizontal_spacing=0.10,
    vertical_spacing=0.12,
)

# CVD-friendly colors
set_colors = {
    "GFI": "#0072B2",
    "GFI_dark": "#009E73",
    "Humbird normal": "#D55E00",
    "Humbird enhanced": "#8C3B00",
}

name_1 = "1. Constant; Humbird (2020)"
name_2 = "2. Constant; Humbird (2020) Enhanced"
name_3 = "3. Constant; Frohlich et al. (2025)"
# name_4 = "4. Dynamic; Frohlich et al. (2025)"
name_5 = "4. Excess, Zeng & Deckwer (1995); Frohlich et al. (2025)"
name_6 = "5. Excess, Zeng & Deckwer (1995) enhanced; Frohlich et al. (2025)"

constant_trace_names = {
    "Humbird normal": name_1,
    "Humbird enhanced": name_2,
    "GFI": name_3,
}

# Row-wise legend draw order
legend_rank = {
    name_1: 1,
    name_3: 2,
    name_5: 3,
    name_2: 4,
    # name_4: 5,
    name_6: 5,
}

linewidth = 5
axis_line_width = 3.5
tick_width = 2.5

for i, (label, idx) in enumerate(yield_panels):
    row = i // 2 + 1
    col = i % 2 + 1

    # Constant traces for all parameter families (solid)
    for set_name in ["Humbird normal", "Humbird enhanced", "GFI"]:
        color = set_colors[set_name]
        trace_name = constant_trace_names[set_name]

        fig_gln.add_trace(
            go.Scatter(
                x=C_Gln_range,
                y=np.full_like(C_Gln_range, constant_yields_gln[set_name][idx]),
                mode="lines",
                name=trace_name,
                legendrank=legend_rank[trace_name],
                line=dict(color=color, width=linewidth, dash="solid"),
                showlegend=(i == 0),
            ),
            row=row,
            col=col,
        )

    # Dynamic trace only for GFI (dashed) - Campaign 3 excluded
    # fig_gln.add_trace(
    #     go.Scatter(
    #         x=C_Gln_range,
    #         y=np.full_like(C_Gln_range, dynamic_yields_gln["GFI"][idx]),
    #         mode="lines",
    #         name=name_4,
    #         legendrank=legend_rank[name_4],
    #         line=dict(color=set_colors["GFI"], width=linewidth, dash="dash"),
    #         showlegend=(i == 0),
    #     ),
    #     row=row,
    #     col=col,
    # )

    fig_gln.add_trace(
        go.Scatter(
            x=C_Gln_range,
            y=excess_normal_yields_matrix_gln[:, idx],
            mode="lines",
            name=name_5,
            legendrank=legend_rank[name_5],
            line=dict(color=set_colors["GFI"], width=linewidth, dash="dot"),
            showlegend=(i == 0),
        ),
        row=row,
        col=col,
    )

    fig_gln.add_trace(
        go.Scatter(
            x=C_Gln_range,
            y=excess_adjusted_yields_matrix_gln[:, idx],
            mode="lines",
            name=name_6,
            legendrank=legend_rank[name_6],
            line=dict(color=set_colors["GFI_dark"], width=linewidth, dash="dot"),
            showlegend=(i == 0),
        ),
        row=row,
        col=col,
    )

    fig_gln.update_xaxes(title_text="C<sub>Gln</sub> (mmol/L)", row=row, col=col)
    fig_gln.update_yaxes(title_text=label, row=row, col=col)

fig_gln.update_xaxes(
    showline=True,
    linewidth=axis_line_width,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=tick_width,
)
fig_gln.update_yaxes(
    showline=True,
    linewidth=axis_line_width,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=tick_width,
)

fig_gln.update_layout(
    width=1700,
    height=1700,
    template="simple_white",
    font=dict(size=35, color="black"),
    legend=dict(
        title_text="",
        orientation="h",
        yanchor="top",
        y=-0.12,
        xanchor="left",
        x=0.0,
        traceorder="normal",
        entrywidthmode="fraction",
        entrywidth=1 / 3,
        font=dict(size=20),
    ),
    margin=dict(l=100, r=0, t=0, b=280),
)

fig_gln.show()


# Update to transparent background
fig_gln.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",   # whole canvas
    plot_bgcolor="rgba(0,0,0,0)",    # plotting area
    legend=dict(bgcolor="rgba(0,0,0,0)")
)

# -----------------------------------------------------------------------------
# 5) Export for publication/reproducibility
# -----------------------------------------------------------------------------
SAVE_EXPORTS = True

if SAVE_EXPORTS == True:

    out_dir = Path.cwd()
    # csv_path = out_dir / "fedbatch_strategy_comparison_metrics.csv"
    svg_path = out_dir / "Kinetic_Profiles_Gln.svg"
    # png_path = out_dir / "fedbatch_strategy_comparison.png"

    # metrics_df.to_csv(csv_path, index=False)
    # print(f"Saved metrics table: {csv_path}")

    try:
        fig_gln.write_image(str(svg_path))
        # fig.write_image(str(png_path), scale=3)
        print(f"Saved figure: {svg_path}")
        # print(f"Saved figure: {png_path}")
    except Exception as exc:
        print("Image export skipped (install/enable kaleido if needed).")
        print(f"Export error: {exc}")

Saved figure: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Method Figures\Kinetic_Profiles_Gln.svg


## 3D plots of excess kinetics as function of C_Glc and C_Gln. 

In [9]:
# 3D surfaces: stoichiometric yields vs glucose and glutamine concentrations

# Grid definition
C_Glc_surface = np.linspace(0.0, 50.0, 80)
C_Gln_surface = np.linspace(0.0, 10.0, 80)
X_glc, Y_gln = np.meshgrid(C_Glc_surface, C_Gln_surface)

# Keep same growth-rate condition as above
mu_fixed = 0.8 * float(params_gfi.cell_properties.mu_max)

# Rebuild excess-coupled models for clarity
model_excess_normal_3d = StoichiometricModel(params_exc_normal, mode=StoichiometryMode.EXCESS_COUPLED)
kin_excess_normal_3d = create_kinetic_model(
    Kinetics.EXCESS,
    params_exc_normal,
    stoichiometric_model=model_excess_normal_3d,
)

model_excess_adjusted_3d = StoichiometricModel(params_exc_adjusted, mode=StoichiometryMode.EXCESS_COUPLED)
kin_excess_adjusted_3d = create_kinetic_model(
    Kinetics.EXCESS,
    params_exc_adjusted,
    stoichiometric_model=model_excess_adjusted_3d,
)

# Yield indices (same 4 panels)
yield_panels_3d = [
    ("YGlc/X", 0),
    ("YLac/Glc", 1),
    ("YGln/Glc", 2),
    ("YAmm/Gln", 6),
]


def build_yield_surfaces(kin_model, stoich_model):
    """Return dict: panel label -> 2D Z surface for that yield index."""
    z_surfaces = {label: np.zeros_like(X_glc, dtype=float) for label, _ in yield_panels_3d}

    for i in range(X_glc.shape[0]):
        for j in range(X_glc.shape[1]):
            c_glc = X_glc[i, j]
            c_gln = Y_gln[i, j]
            cat_yields = kin_model.calculate_cat_yields_from_excess(mu_fixed, c_glc, c_gln)
            yields_vec = stoich_model.calculate_yields(mu_fixed, cat_yields=cat_yields)
            for label, idx in yield_panels_3d:
                z_surfaces[label][i, j] = yields_vec[idx]

    return z_surfaces


z_normal = build_yield_surfaces(kin_excess_normal_3d, model_excess_normal_3d)
z_adjusted = build_yield_surfaces(kin_excess_adjusted_3d, model_excess_adjusted_3d)


def make_surface_figure(z_data, title):
    fig_surface = make_subplots(
        rows=2,
        cols=2,
        specs=[[{"type": "surface"}, {"type": "surface"}], [{"type": "surface"}, {"type": "surface"}]],
        subplot_titles=[label for label, _ in yield_panels_3d],
        horizontal_spacing=0.03,
        vertical_spacing=0.08,
    )

    for k, (label, _) in enumerate(yield_panels_3d):
        row = k // 2 + 1
        col = k % 2 + 1
        fig_surface.add_trace(
            go.Surface(
                x=X_glc,
                y=Y_gln,
                z=z_data[label],
                colorscale="Viridis",
                showscale=(k == 0),
                colorbar=dict(title="Yield", len=0.45, y=0.78),
                name=label,
                hovertemplate=(
                    "Glucose: %{x:.2f} mmol/L<br>"
                    "Glutamine: %{y:.2f} mmol/L<br>"
                    f"{label}: "
                    "%{z:.4f}<extra></extra>"
                ),
            ),
            row=row,
            col=col,
        )

    fig_surface.update_layout(
        title=title,
        width=1200,
        height=1000,
        template="simple_white",
        font=dict(family="Arial", size=12, color="black"),
        margin=dict(l=20, r=20, t=70, b=20),
    )

    # Set per-panel 3D axis labels
    axis_line_width_3d = 5
    for idx_scene in range(1, 5):
        scene_name = "scene" if idx_scene == 1 else f"scene{idx_scene}"
        fig_surface.layout[scene_name].xaxis.title = "Glucose concentration (mmol/L)"
        fig_surface.layout[scene_name].yaxis.title = "Glutamine concentration (mmol/L)"
        fig_surface.layout[scene_name].zaxis.title = "Stoichiometric yield coefficient"

        fig_surface.layout[scene_name].xaxis.showline = True
        fig_surface.layout[scene_name].yaxis.showline = True
        fig_surface.layout[scene_name].zaxis.showline = True

        fig_surface.layout[scene_name].xaxis.linecolor = "black"
        fig_surface.layout[scene_name].yaxis.linecolor = "black"
        fig_surface.layout[scene_name].zaxis.linecolor = "black"

        fig_surface.layout[scene_name].xaxis.linewidth = axis_line_width_3d
        fig_surface.layout[scene_name].yaxis.linewidth = axis_line_width_3d
        fig_surface.layout[scene_name].zaxis.linewidth = axis_line_width_3d

    return fig_surface


fig_surface_normal = make_surface_figure(
    z_normal,
    "3D stoichiometric yield surfaces — Excess-coupled model (Normal excess kinetics)",
)
fig_surface_adjusted = make_surface_figure(
    z_adjusted,
    "3D stoichiometric yield surfaces — Excess-coupled model (Adjusted excess kinetics)",
)

fig_surface_normal.show()
fig_surface_adjusted.show()